In [1]:
import pandas as pd
import io
import glob
import os

### Let's look at our data so we can decide what to keep in our final CSV and how to format it

In [2]:
#file_path = 'stats\\1 wall 2 targets horizontal - Challenge - 2020.07.30-18.51.11 Stats.csv'
file_path = 'stats\\waldoTS Novice - Challenge - 2026.05.19-20.45.52 Stats.csv'

Each CSV contains 4 tables (each effectively its own csv), so we need to split them up

In [3]:
# open and split the file contents by empty lines
with open(file_path) as f:
    table_chunks = f.read().split("\n\n") 

# Load each block into its own DataFrame
kill_log_df = pd.read_csv(io.StringIO(table_chunks[0]))
weapon_info_df = pd.read_csv(io.StringIO(table_chunks[1]))
stats_df = pd.read_csv(io.StringIO(table_chunks[2]))
config_df = pd.read_csv(io.StringIO(table_chunks[3]))


#### The first table is the kill log, unlikely we will keep any of this at this time

In [4]:
kill_log_df.head()

,Kill #,Timestamp,Bot,Weapon,TTK,Shots,Hits,Accuracy,Damage Done,Damage Possible,Efficiency,Cheated,OverShots
0,1,20:44:53.437,Quaker Bot Long Strafes,LG,0.673000s,69,24,0.347826,120.0,345.0,0.347826,0,10
1,2,20:44:53.827,Quaker Bot Long Strafes,LG,0.230000s,24,24,1.000000,120.0,120.0,1.000000,0,15
2,3,20:44:54.188,Quaker Bot Long Strafes,LG,0.230000s,24,24,1.000000,120.0,120.0,1.000000,0,12
3,4,20:44:54.588,Quaker Bot Long Strafes,LG,0.229000s,24,24,1.000000,120.0,120.0,1.000000,0,16
4,5,20:44:54.987,Quaker Bot Long Strafes,LG,0.230000s,24,24,1.000000,120.0,120.0,1.000000,0,16


#### The second table gives us shots & hits but those are also present in the stats table

In [5]:
weapon_info_df.head()

,Weapon,Shots,Hits,Damage Done,Damage Possible,Unnamed: 5,Sens Scale,Horiz Sens,Vert Sens,FOV,Hide Gun,Crosshair,Crosshair Scale,Crosshair Color,ADS Sens,ADS Zoom Scale,Avg Target Scale,Avg Time Dilation
0,LG,3747,2963,14815.0,18735.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### The third table gives us most of what we want, it just isn't formatted well and will have to be manipulated

In [6]:
stats_df.head()

,Kills:,123
0,Deaths:,0
1,Fight Time:,36.123001
2,Time Remaining:,0.0
3,Avg TTK:,0.487744
4,Damage Done:,14815.0


#### The fourth table has info such as sensitivity and FOV, also isn't formatted well but we might as well include it in our CSV

In [7]:
config_df.head()

,Input Lag:,0
0,Max FPS (config):,999.0
1,Sens Scale:,cm/360
2,Sens Increment:,0.157005
3,Horiz Sens:,52.0
4,Vert Sens:,52.0


In [8]:
stats_df_T = stats_df.set_index(stats_df.columns[0]).T
stats_df_T

Kills:,Deaths:,Fight Time:,Time Remaining:,Avg TTK:,Damage Done:,Total Overshots:,Damage Taken:,Hit Count:,Miss Count:,Midairs:,...,MBS Points:,Score:,Scenario:,Hash:,Game Version:,Challenge Start:,Pause Count:,Pause Duration:,Avg Target Scale:,Avg Time Dilation:
123,0,36.123001,0.0,0.487744,14815.0,2242,0.0,2963,784,0,...,0.0,148.149994,waldoTS Novice,df14bebca22672e9e931d983893ce8ce,3.9.1.2026-05-06-17-06-54-4967be094932,20:44:52.651,0,0,1.0,1.0


In [9]:
config_df_T = config_df.set_index(config_df.columns[0]).T
config_df_T

Input Lag:,Max FPS (config):,Sens Scale:,Sens Increment:,Horiz Sens:,Vert Sens:,DPI:,FOV:,FOVScale:,Hide Gun:,Crosshair:,Crosshair Scale:,Crosshair Color:,Resolution:,Avg FPS:,Resolution Scale:
0,999.0,cm/360,0.157005,52.0,52.0,1600,103.0,Valorant,false,aimb.png,1.0,000001FF,1920x1080,826.00293,100.0


In [10]:
combined_df = pd.concat([stats_df_T.reset_index(drop=True), config_df_T.reset_index(drop=True)], axis=1)
combined_df

,Deaths:,Fight Time:,Time Remaining:,Avg TTK:,Damage Done:,Total Overshots:,Damage Taken:,Hit Count:,Miss Count:,Midairs:,...,DPI:,FOV:,FOVScale:,Hide Gun:,Crosshair:,Crosshair Scale:,Crosshair Color:,Resolution:,Avg FPS:,Resolution Scale:
0,0,36.123001,0.0,0.487744,14815.0,2242,0.0,2963,784,0,...,1600,103.0,Valorant,false,aimb.png,1.0,000001FF,1920x1080,826.00293,100.0


In [11]:
# remove the last character ':' from every column name
combined_df.columns = [col[:-1] for col in combined_df.columns]
combined_df.head()

,Deaths,Fight Time,Time Remaining,Avg TTK,Damage Done,Total Overshots,Damage Taken,Hit Count,Miss Count,Midairs,...,DPI,FOV,FOVScale,Hide Gun,Crosshair,Crosshair Scale,Crosshair Color,Resolution,Avg FPS,Resolution Scale
0,0,36.123001,0.0,0.487744,14815.0,2242,0.0,2963,784,0,...,1600,103.0,Valorant,false,aimb.png,1.0,000001FF,1920x1080,826.00293,100.0


In [12]:
combined_df.to_csv('combined_stats_config.csv', index=False)

### Now let's iterate through every file

In [13]:
# get all CSV files in the stats folder
stats_dir = 'stats'
all_files = sorted(glob.glob(os.path.join(stats_dir, '*.csv')))
print(f"Found {len(all_files)} files to process")

Found 11454 files to process


In [14]:
# Process all files and combine into one dataframe
import re
from datetime import datetime

all_data = []

for file_path in all_files:
    try:
        # Open and split the file contents by empty lines
        with open(file_path) as f:
            table_chunks = f.read().split("\n\n")
        
        # Load each block into its own DataFrame
        stats_df_temp = pd.read_csv(io.StringIO(table_chunks[2]))
        config_df_temp = pd.read_csv(io.StringIO(table_chunks[3]))
        
        # Transpose both dataframes
        stats_df_T = stats_df_temp.set_index(stats_df_temp.columns[0]).T
        config_df_T = config_df_temp.set_index(config_df_temp.columns[0]).T
        
        # Combine stats and config
        combined = pd.concat([stats_df_T.reset_index(drop=True), config_df_T.reset_index(drop=True)], axis=1)
        
        # remove the character ':' from every column name
        combined.columns = [col.rstrip(':') for col in combined.columns]
        
        # Add the filename as metadata
        filename = os.path.basename(file_path)
        combined['filename'] = filename
        
        # Extract timestamp from filename (format: YYYY.MM.DD-HH.MM.SS)
        timestamp_match = re.search(r'(\d{4}\.\d{2}\.\d{2})-(\d{2}\.\d{2}\.\d{2})', filename)
        if timestamp_match:
            date_part = timestamp_match.group(1)  # YYYY.MM.DD
            time_part = timestamp_match.group(2)  # HH.MM.SS
            # Convert dots to colons/dashes for datetime parsing
            datetime_str = date_part.replace('.', '-') + ' ' + time_part.replace('.', ':')
            combined['timestamp'] = pd.to_datetime(datetime_str)
        else:
            combined['timestamp'] = pd.NaT
        
        all_data.append(combined)
        print(f"Processed: {filename}")
        
    except Exception as e:
        print(f"Error with {os.path.basename(file_path)}: {str(e)}")

# Combine all dataframes into one
final_df = pd.concat(all_data, ignore_index=True)
print(f"\nTotal records: {len(final_df)}")
print(f"Columns: {list(final_df.columns)}")

Processed: (Survival) Apex Legends - Advanced Aim Training Series - Challenge - 2019.05.09-21.02.37 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.32.31 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.33.15 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.33.42 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.34.08 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.34.34 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.35.08 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.35.56 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.36.24 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.36.59 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.37.29 Stats.csv
Processed: 1 Wall 6 targets Adjust - Challenge - 2020.08.05-14.38.16 Stats.csv
Processed: 1 Wall 6 t

In [15]:
final_df.head()

,Deaths,Fight Time,Avg TTK,Damage Done,Damage Taken,Midairs,Midaired,Directs,Directed,Distance Traveled,...,DPI,FOVScale,Time Remaining,Total Overshots,Hit Count,Miss Count,MBS Points,Avg Target Scale,Avg Time Dilation,Sens Increment
0,0,9.827,4.913,494.0,223.199997,0,0,0,0,9309.963867,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0,7.11,0.465,1075.0,0.0,0,0,0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0,6.893,0.392,1275.0,0.0,0,0,0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0,5.953,0.385,1300.0,0.0,0,0,0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0,6.91,0.37,1350.0,0.0,0,0,0,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# Save the combined dataset
output_file = 'all_stats_combined.csv'
final_df.to_csv(output_file, index=False)

In [17]:
# Verify the timestamp column was created properly
print("Columns in final_df:")
print(list(final_df.columns))
print("\n" + "="*50 + "\n")
print("Sample of filename and timestamp columns:")
print(final_df[['filename', 'timestamp']].head(10))

Columns in final_df:
['Deaths', 'Fight Time', 'Avg TTK', 'Damage Done', 'Damage Taken', 'Midairs', 'Midaired', 'Directs', 'Directed', 'Distance Traveled', 'Score', 'Scenario', 'Hash', 'Game Version', 'Max FPS (config)', 'Sens Scale', 'Horiz Sens', 'Vert Sens', 'FOV', 'Hide Gun', 'Crosshair', 'Crosshair Scale', 'Crosshair Color', 'filename', 'timestamp', 'Reloads', 'Challenge Start', 'Pause Count', 'Pause Duration', 'Resolution', 'Avg FPS', 'Resolution Scale', 'DPI', 'FOVScale', 'Time Remaining', 'Total Overshots', 'Hit Count', 'Miss Count', 'MBS Points', 'Avg Target Scale', 'Avg Time Dilation', 'Sens Increment']


Sample of filename and timestamp columns:
                                            filename           timestamp
0  (Survival) Apex Legends - Advanced Aim Trainin... 2019-05-09 21:02:37
1  1 Wall 6 targets Adjust - Challenge - 2020.08.... 2020-08-05 14:32:31
2  1 Wall 6 targets Adjust - Challenge - 2020.08.... 2020-08-05 14:33:15
3  1 Wall 6 targets Adjust - Challenge - 202

In [18]:
# print the data in a datatime range from the month of May 2026 as an example
may_2026_data = final_df[(final_df['timestamp'] >= '2026-05-01') & (final_df['timestamp'] < '2026-06-01')]
print("Data from May 2026:")
print(may_2026_data[['filename', 'timestamp']])

Data from May 2026:
                                                filename           timestamp
285    1w2ts reload smallflicks - Challenge - 2026.05... 2026-05-29 19:19:04
286    1w2ts reload smallflicks larger - Challenge - ... 2026-05-11 21:04:25
287    1w2ts reload smallflicks larger - Challenge - ... 2026-05-11 21:06:43
288    1w2ts reload smallflicks larger - Challenge - ... 2026-05-11 21:09:00
289    1w2ts reload smallflicks larger - Challenge - ... 2026-05-11 21:10:46
...                                                  ...                 ...
11312  waldoTS Novice - Challenge - 2026.05.19-20.47.... 2026-05-19 20:47:31
11356  ww3t Voltaic - Challenge - 2026.05.29-19.16.05... 2026-05-29 19:16:05
11439  ww3t Voltaic 50% smaller - Challenge - 2026.05... 2026-05-29 19:17:41
11447  ww5t Voltaic Slightly Larger - Challenge - 202... 2026-05-16 18:40:15
11448  ww5t Voltaic Slightly Larger - Challenge - 202... 2026-05-16 20:01:46

[709 rows x 2 columns]


In [19]:
final_df.columns

Index(['Deaths', 'Fight Time', 'Avg TTK', 'Damage Done', 'Damage Taken',
       'Midairs', 'Midaired', 'Directs', 'Directed', 'Distance Traveled',
       'Score', 'Scenario', 'Hash', 'Game Version', 'Max FPS (config)',
       'Sens Scale', 'Horiz Sens', 'Vert Sens', 'FOV', 'Hide Gun', 'Crosshair',
       'Crosshair Scale', 'Crosshair Color', 'filename', 'timestamp',
       'Reloads', 'Challenge Start', 'Pause Count', 'Pause Duration',
       'Resolution', 'Avg FPS', 'Resolution Scale', 'DPI', 'FOVScale',
       'Time Remaining', 'Total Overshots', 'Hit Count', 'Miss Count',
       'MBS Points', 'Avg Target Scale', 'Avg Time Dilation',
       'Sens Increment'],
      dtype='str')

In [20]:
final_df_cleaned = final_df[['Fight Time', 'Avg TTK', 'Damage Done', 
       'Score', 'Scenario', 'Hash', 'Game Version', 'Max FPS (config)',
       'Sens Scale', 'Horiz Sens', 'Vert Sens', 'FOV',
       'filename', 'timestamp',
       'Reloads', 'Challenge Start', 'Pause Count', 'Pause Duration',
       'Resolution', 'Avg FPS', 'Resolution Scale', 'DPI', 'FOVScale',
       'Time Remaining', 'Total Overshots', 'Hit Count', 'Miss Count',
       ]]
final_df_cleaned.head()

,Fight Time,Avg TTK,Damage Done,Score,Scenario,Hash,Game Version,Max FPS (config),Sens Scale,Horiz Sens,...,Pause Duration,Resolution,Avg FPS,Resolution Scale,DPI,FOVScale,Time Remaining,Total Overshots,Hit Count,Miss Count
0,9.827,4.913,494.0,363.673462,(Survival) Apex Legends - Advanced Aim Trainin...,b3aa5f432edf806c573d26f27e17f902,1.0.7.2,200.0,Fortnite Slider,0.1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7.11,0.465,1075.0,2759.701416,1 Wall 6 targets Adjust,62aa01b4e81e6887fbb38788b335ed43,2.0.2.0,300.0,Valorant,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,6.893,0.392,1275.0,3468.0,1 Wall 6 targets Adjust,62aa01b4e81e6887fbb38788b335ed43,2.0.2.0,300.0,Valorant,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5.953,0.385,1300.0,3755.55542,1 Wall 6 targets Adjust,62aa01b4e81e6887fbb38788b335ed43,2.0.2.0,300.0,Valorant,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6.91,0.37,1350.0,3888.000244,1 Wall 6 targets Adjust,62aa01b4e81e6887fbb38788b335ed43,2.0.2.0,300.0,Valorant,0.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
# new csv file with only the columns we care about
final_df_cleaned.to_csv('all_stats_combined_cleaned.csv', index=False)